# 10 - Data Cleaning & Text Preprocessing (Production Dataset)
## ShopEase Europe | Sentiment Analysis Project - Phase 2 
**Objective:** Clean the production dataset, engineer temporal and 
text-based features, and preprocess review text into a model-ready 
format for both classical and transformer-based modelling.

## Import Libaries

In [1]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## Load the Dataset

In [2]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', 'amazon_reviews_cleaned.csv')
PROCESSED_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'production_clean_reviews.csv')

df = pd.read_csv(RAW_DATA_PATH)

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Dataset loaded: 21,055 rows x 7 columns


## Structural Cleaning
Standardising sentiment casing, handling the missing country value, 
and converting timestamp to a proper datetime format.

In [4]:
# Standardise sentiment to lowercase
df['sentiment'] = df['sentiment'].str.lower()

# Handle missing country
print(f"Missing country values before: {df['country'].isnull().sum()}")
df['country'] = df['country'].fillna('Unknown')
print(f"Missing country values after: {df['country'].isnull().sum()}")

# Convert timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed')

print(f"\nSentiment values: {df['sentiment'].unique()}")
print(f"Timestamp dtype: {df['timestamp'].dtype}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

Missing country values before: 0
Missing country values after: 0

Sentiment values: ['negative' 'positive' 'neutral']
Timestamp dtype: datetime64[ns, UTC]
Date range: 2007-08-27 17:25:01+00:00 to 2024-09-17 13:19:27+00:00


## Structural Cleaning Finding

The dataset spans 17 years, from August 2007 to September 2024. This 
wide range likely reflects genuine historical Amazon review data 
spanning many years rather than data generated for a specific recent 
campaign or product launch period.

One missing country value was identified and filled as "Unknown" 
rather than dropped, preserving the review for sentiment and text 
analysis while excluding it from country-specific breakdowns.

## Text Cleaning

In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[!]{2,}', '!', text)
    text = re.sub(r'[?]{2,}', '?', text)
    text = re.sub(r'[.]{2,}', '.', text)
    text = re.sub(r'[^a-zA-Z0-9\s\.,!?\'-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_review'] = df['review'].apply(clean_text)

print(f"Sample original:\n{df['review'].iloc[0][:200]}")
print(f"\nSample cleaned:\n{df['cleaned_review'].iloc[0][:200]}")
print(f"\nOriginal length avg:  {df['review'].str.len().mean():.1f} characters")
print(f"Cleaned length avg:   {df['cleaned_review'].str.len().mean():.1f} characters")

Sample original:
I registered on the website, tried to order a laptop, entered all the details, but instead of charging me and sending the product, they froze my account, demanding various verification documents. I se

Sample cleaned:
i registered on the website, tried to order a laptop, entered all the details, but instead of charging me and sending the product, they froze my account, demanding various verification documents. i se

Original length avg:  460.8 characters
Cleaned length avg:   457.3 characters


## Text Cleaning Finding

Average review length dropped only slightly from 460.8 to 457.3 
characters after cleaning, indicating minimal noise such as URLs, 
excessive punctuation, or special characters in the original text. 
This is expected for genuine, manually written reviews compared to 
more heavily formatted or templated text.

## Feature Engineering
Deriving temporal and text-based features to support analysis and 
modelling.

### Temporal Features
Extracting year, month and month name from the timestamp to support 
time based trend analysis.

In [6]:
df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['month_name'] = df['timestamp'].dt.strftime('%B')

print(f"Year range: {df['year'].min()} to {df['year'].max()}")
print(f"\nReviews per year:")
print(df['year'].value_counts().sort_index())

Year range: 2007 to 2024

Reviews per year:
year
2007       1
2008       3
2009       9
2010      22
2011     360
2012    1166
2013      88
2014     166
2015     175
2016     317
2017     616
2018    1041
2019    2314
2020    2638
2021    2750
2022    2551
2023    4017
2024    2821
Name: count, dtype: int64


## Temporal Feature Finding

Reviews are heavily concentrated in recent years, with 2019 through 
2024 accounting for the majority of the dataset. The earliest years, 
2007 through 2017, contribute relatively few reviews, ranging from 
just 1 in 2007 to several hundred per year by the mid 2010s.

One anomaly stands out, 2012 shows 1,166 reviews, noticeably higher 
than the surrounding years of 88 in 2013 and 360 in 2011. This spike 
will be investigated further during exploratory analysis rather than 
assumed to be a data quality issue at this stage.

**Insight:** Given the sparse early year coverage, any time based 
trend analysis should focus primarily on 2019 onward, where sample 
sizes are large enough to support reliable conclusions.

#### Text Length Features

In [8]:
df['review_length'] = df['cleaned_review'].str.len()
df['word_count'] = df['cleaned_review'].str.split().str.len()

print("Review length and word count by sentiment:")
print(df.groupby('sentiment')[['review_length', 'word_count']].mean().round(2))

Review length and word count by sentiment:
           review_length  word_count
sentiment                           
negative          564.86      104.73
neutral           379.38       70.60
positive          203.85       37.10


## Text Length Finding

Review length shows strong variation by sentiment. Negative reviews 
average 564.86 characters and 104.73 words, more than double the 
length of positive reviews at 203.85 characters and 37.10 words. 
Neutral reviews sit in between, at 379.38 characters and 70.6 words.

**Insight:** This pattern reflects genuine customer behaviour, 
dissatisfied customers tend to write detailed explanations of what 
went wrong, while satisfied customers often leave brief, simple praise. 
This makes review length a potentially useful feature for classical 
modelling.

## Tokenisation and Stop Word Removal
Since this dataset is confirmed monolingual English, preprocessing 
is simplified compared to multilingual handling, using a single 
English stop word list throughout.

In [9]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)

english_stops = set(stopwords.words('english'))

def tokenise_and_clean(text):
    tokens = str(text).lower().split()
    tokens = [t.strip('.,!?;:\'"()[]') for t in tokens]
    tokens = [t for t in tokens if t not in english_stops and len(t) > 2]
    return tokens

df['tokens'] = df['cleaned_review'].apply(tokenise_and_clean)

print(f"Sample tokens: {df['tokens'].iloc[0][:15]}")
print(f"\nAverage tokens per review: {df['tokens'].apply(len).mean():.1f}")

Sample tokens: ['registered', 'website', 'tried', 'order', 'laptop', 'entered', 'details', 'instead', 'charging', 'sending', 'product', 'froze', 'account', 'demanding', 'various']

Average tokens per review: 40.9


## Lemmatisation

In [10]:
import spacy

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

def lemmatise(tokens):
    text = ' '.join(tokens)
    doc = nlp(text)
    return [token.lemma_ for token in doc]

df['lemmatised'] = df['tokens'].apply(lemmatise)

print("Lemmatisation complete")
print(f"\nSample before: {df['tokens'].iloc[0][:10]}")
print(f"Sample after:  {df['lemmatised'].iloc[0][:10]}")

Lemmatisation complete

Sample before: ['registered', 'website', 'tried', 'order', 'laptop', 'entered', 'details', 'instead', 'charging', 'sending']
Sample after:  ['register', 'website', 'try', 'order', 'laptop', 'enter', 'detail', 'instead', 'charge', 'send']


## TF-IDF Vectorisation

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

df['preprocessed_text'] = df['lemmatised'].apply(lambda tokens: ' '.join(tokens))

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

tfidf_matrix = tfidf.fit_transform(df['preprocessed_text'])

print(f"Vocabulary size: {len(tfidf.get_feature_names_out()):,} features")
print(f"Matrix shape: {tfidf_matrix.shape}")
print(f"\nSample features: {list(tfidf.get_feature_names_out()[:20])}")

Vocabulary size: 10,000 features
Matrix shape: (21055, 10000)

Sample features: ['00', '00 amazon', '00 dollar', '000', '00pm', '01', '07', '10', '10 000', '10 day', '10 pm', '100', '1000', '10th', '11', '111', '112', '113', '114', '119']


## TF-IDF Finding

The vocabulary reached the maximum cap of 10,000 features, suggesting 
the underlying text contains substantially more unique vocabulary than 
this limit captures. Sample features include specific numbers, times 
and dollar amounts, reflecting the detailed, concrete language customers 
use when describing real transactions and experiences, such as order 
numbers, prices and timestamps.

**Insight:** This rich, varied vocabulary gives classical models genuine 
signal to learn from, unlike a constrained template vocabulary. We expect 
classical model performance in the next notebook to reflect realistic, 
imperfect accuracy rather than artificially perfect scores.

In [ ]:
import pickle
from scipy.sparse import save_npz

PROCESSED_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed')

df[['review_id', 'cleaned_review', 'preprocessed_text', 'sentiment', 
    'country', 'product_category', 'year', 'month', 'rating',
    'review_length', 'word_count']].to_csv(
    os.path.join(PROCESSED_PATH, 'production_preprocessed_reviews.csv'), index=False)

save_npz(os.path.join(PROCESSED_PATH, 'production_tfidf_matrix.npz'), tfidf_matrix)

with open(os.path.join(PROCESSED_PATH, 'production_tfidf_vectoriser.pkl'), 'wb') as f:
    pickle.dump(tfidf, f)

print("Outputs saved successfully")
print(f"Preprocessed dataset: {df.shape[0]:,} rows")
print(f"TF-IDF matrix: {tfidf_matrix.shape}")

Outputs saved successfully
Preprocessed dataset: 21,055 rows
TF-IDF matrix: (21055, 10000)


: 

## Summary

The production dataset was cleaned, enriched with temporal and text 
length features, and preprocessed through tokenisation, stop word 
removal, lemmatisation and TF-IDF vectorisation.

Review length emerged as a genuinely useful signal, negative reviews 
average more than double the length of positive ones, reflecting how 
dissatisfied customers tend to write detailed complaints while satisfied 
customers leave brief praise. The vocabulary reached the maximum 10,000 
feature cap, indicating substantially richer language than a constrained 
or templated dataset would produce.

These findings give classical models in the next notebook genuine 
signal to learn from, setting up a meaningful comparison against 
DistilBERT fine-tuning later in this phase.

### Business Impact
The strong relationship between review length and sentiment offers 
ShopEase Europe a simple, low cost early warning signal, longer reviews 
flagged for sentiment analysis could be automatically prioritised for 
faster customer service review, since length alone correlates with the 
likelihood of a negative experience requiring attention.

Outputs saved to data/processed/:
- production_clean_reviews.csv
- production_preprocessed_reviews.csv
- production_tfidf_matrix.npz
- production_tfidf_vectoriser.pkl